Se o Algoritmo Genético tiver, por exemplo, 100 indivíduos, rodando por 500 gerações, ele vai precisar calcular a distância entre cidades muitas vezes. Se for usar a Fórmula de Haversine (que envolve senos, cossenos e raízes quadradas) toda vez que for avaliar a aptidão de um indivíduo (vou usar), o código vai ficar extremamente lento.

A solução é criar uma Matriz de Distâncias (33x33) com todas as distancias pré-calculadas, pra que o algoritmo precise apenas consultar essa matriz, ao invés de ficar fazendo inúmeros calculos.

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import haversine_distances

# carregando o dataset limpo ('dataset_final_clean')
aeroportos = pd.read_csv("dataset_final_clean.csv")

# isolando apenas as colunas de coordenadas
coordenadas = aeroportos[['latitude_deg', 'longitude_deg']].values

# convertendo de graus para radianos (aparentemente é exigência da biblioteca scikit-learn) (a função pronta de haversine vem de lá)
coordenadas_rad = np.radians(coordenadas)

# calculando a matriz de distâncias (o resultado sai em radianos)
matriz_dist_rad = haversine_distances(coordenadas_rad, coordenadas_rad)

# multiplicando pelo raio médio da Terra (em km)
raio_terra_km = 6371.0088
matriz_dist_km = matriz_dist_rad * raio_terra_km

# transformando a matriz numpy em um dataframe do pandas
# para ficar legível e fácil de consultar depois no algoritmo genético
matriz_distancias = pd.DataFrame(
    matriz_dist_km, 
    index=aeroportos['ident'],    # nomeia as linhas com o código do aeroporto ('ident')
    columns=aeroportos['ident']   # nomeia as colunas com o código do aeroporto ('ident')
)

# mostrando o inicio da matriz para verificação
display(matriz_distancias.head())

ident,SBBE,SBBR,SBBV,SBCF,SBCT,SBCY,SBEG,SBFI,SBFL,SBFZ,...,SBSG,SBSL,SBSP,SBSV,SBVT,SBMQ,SBCG,SBTE,SBAR,SBPJ
ident,,,,,,,,,,,,,,,,,,,,,
SBBE,0.000000,1612.354114,1436.761709,2088.486277,2686.321583,1794.743872,1299.080744,2768.627131,2923.441145,1136.212080,...,1534.260797,490.049538,2481.686987,1701.195824,2280.129543,329.532879,2226.971517,749.130690,1651.016193,991.082678
SBBR,1612.354114,0.000000,2510.483471,592.155615,1081.947106,877.349886,1948.612632,1278.408066,1313.837345,1691.763034,...,1770.735988,1531.120580,872.803051,1084.754467,943.000365,1803.684360,877.696570,1324.451925,1292.457530,622.004035
SBBV,1436.761709,2510.483471,0.000000,3096.248146,3390.318153,2117.642631,658.222747,3232.227483,3633.749236,2570.183943,...,2971.017414,1926.263274,3312.936803,3027.765396,3399.402062,1113.348930,2674.392371,2170.756104,3031.928447,1999.656626
SBCF,2088.486277,592.155615,3096.248146,0.000000,845.543427,1360.951509,2540.764398,1266.283754,1007.835572,1859.488081,...,1800.703934,1896.024728,523.606157,960.199269,391.012373,2321.952079,1122.031833,1625.440470,1212.795916,1140.782512
SBCT,2686.321583,1081.947106,3390.318153,845.543427,0.000000,1313.747581,2757.789248,533.035262,246.084718,2672.510813,...,2643.043871,2605.252676,331.149534,1805.601491,1082.477661,2851.561365,795.452188,2374.470350,2058.333845,1696.475801


In [5]:
# salvando a matriz de distancias num csv
matriz_distancias.to_csv("matriz_distancias.csv", index=True)

# para carregar esse arquivo no futuro (no notebook do Algoritmo Genético)
# terá que ser assim para que o pandas entenda que a primeira coluna é o nome das linhas e não um dado aleatório:
# matriz = pd.read_csv("matriz_distancias.csv", index_col=0)
# o 'index_col=0' avisa ao pandas que a primeira coluna é o índice de nomes